### Imports

In [1]:
import pandas as pd
import ta
import yfinance as yf

### Get stock data

**Currently a test**

In [2]:
tickers = ["NVDA", "GOOGL", "SPY", "DIA", "QQQ"]

all_data = yf.download(tickers, period="3mo", interval="1d")

for ticker in tickers:
    df = all_data.xs(ticker, axis=1, level=1) # extract one ticker
    
    df['rsi'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
    df['pct_change'] = df['Close'].pct_change() * 100

    print(df[['Close', 'rsi', 'pct_change']].tail())
# data = yf.download(tickers, period="3mo", interval="1d")

# # fix close data typing issue
# if isinstance(data.columns, pd.MultiIndex):
#     data.columns = data.columns.get_level_values(0)
# print(type(data['Close']))
# print(data['Close'].shape)

[*********************100%***********************]  5 of 5 completed

Price            Close        rsi  pct_change
Date                                         
2026-04-27  216.610001  76.564923    4.004416
2026-04-28  213.169998  71.201073   -1.588109
2026-04-29  209.250000  65.564354   -1.838907
2026-04-30  199.570007  54.161699   -4.626042
2026-05-01  198.449997  53.012881   -0.561212
Price            Close        rsi  pct_change
Date                                         
2026-04-27  350.339996  71.756844    1.724739
2026-04-28  349.779999  71.120524   -0.159844
2026-04-29  349.940002  71.199110    0.045744
2026-04-30  384.799988  82.422067    9.961703
2026-05-01  385.690002  82.608402    0.231293
Price            Close        rsi  pct_change
Date                                         
2026-04-27  715.169983  71.080621    0.172281
2026-04-28  711.690002  67.435500   -0.486595
2026-04-29  711.580017  67.318001   -0.015454
2026-04-30  718.659973  70.840179    0.994963
2026-05-01  720.750000  71.806148    0.290823
Price            Close        rsi 


C:\Users\sheen\AppData\Local\Temp\ipykernel_72600\815163727.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rsi'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
C:\Users\sheen\AppData\Local\Temp\ipykernel_72600\815163727.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pct_change'] = df['Close'].pct_change() * 100
C:\Users\sheen\AppData\Local\Temp\ipykernel_72600\815163727.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

### Adding indicators

Right now I am using relative strength index (RSI). RSI is the relative strength index from 0-100. 30- = oversold (price drop), 70+ = overbought. I'm using RSI to detect potential rebounds after drops

Maybe I could use other indicators such as moving average, multiples such as Price to earnings, price to book value, + price to free cash flow

In [ ]:
# data['rsi'] = ta.momentum.RSIIndicator(data['Close'], window=14).rsi()
# data['rsi']

ValueError: Data must be 1-dimensional, got ndarray of shape (63, 5) instead

: 

### Detecting drops in price

In [ ]:
# data['pct_change'] = data['Close'].pct_change() * 100

df['buy_signal'] = (df['pct_change'] <= -3) | (df['rsi'] < 30)

Date
2026-02-02    False
2026-02-03    False
2026-02-04    False
2026-02-05    False
2026-02-06    False
              ...  
2026-04-24    False
2026-04-27    False
2026-04-28    False
2026-04-29    False
2026-04-30    False
Name: buy_signal, Length: 62, dtype: bool

### Demonstrativ backtesting

**Currently spending 5% of balance on each trade**

BUY CONDITIONS:
- Buy signal present, no curent position
then uses all money t buy shares

SELL CONDITIONS:
- take profit @ +2%
- Stop loss @ -30%

In [ ]:
balance = 10000
total_shares = 0
total_cost = 0
risk_percent = 0.05

for i in range(len(data)):
    row = data.iloc[i]
    current_price = row['Close']

    # BUYING
    if row['buy_signal'] and total_shares == 0:
        entry_price = row['Close']
        trade_amount = balance * risk_percent
        shares = trade_amount / entry_price

        total_shares += shares
        total_cost += shares * entry_price
        balance -= shares * entry_price

        print(f"BUY at {entry_price}")

    # SCALING IN
    elif total_shares > 0:
        avg_entry_price = total_cost / total_shares

        drop_from_avg = (row['Close'] - avg_entry_price) / avg_entry_price
        if drop_from_avg <= -0.02 and balance > 0: # 2% lower
            trade_amount = balance * risk_percent
            shares = trade_amount / row['Close']

            total_shares += shares
            total_cost += shares * row['Close']

            balance -= shares * row['Close']

            print(f"Adding at {row['Close']}")

    # SELLING
    elif total_shares > 0:
        avg_entry_price = total_cost / total_shares
        change = (current_price - avg_entry_price) / avg_entry_price

        if change >= 0.02 or change <= -0.30:
            balance = total_shares * current_price
            total_shares = 0
            total_cost = 0
            print(f"SELL at {current_price}")